### Using built-in ase optimization with vasp



ASE has some nice optimization tools built into it. We can use them in vasp too. This example is adapted from this test: [https://trac.fysik.dtu.dk/projects/ase/browser/trunk/ase/test/vasp/vasp_Al_volrelax.py](https://trac.fysik.dtu.dk/projects/ase/browser/trunk/ase/test/vasp/vasp_Al_volrelax.py)

First the VASP way.



In [1]:
from vasp import Vasp
from ase.lattice import bulk

Al = bulk('Al', 'fcc', a=4.5, cubic=True)
calc = Vasp(label='bulk/Al-lda-vasp',
            xc='LDA', isif=7, nsw=5,
            ibrion=1, ediffg=-1e-3,
            lwave=False, lcharg=False,
            atoms=Al)
print(calc.potential_energy)
print(calc)

#+begin_example
-10.07430725

Vasp calculation in /home-research/jkitchin/dft-book/bulk/Al-lda-vasp

INCAR created by Atomic Simulation Environment
 ISIF = 7
 LCHARG = .FALSE.
 IBRION = 1
 EDIFFG = -0.001
 ISMEAR = 1
 LWAVE = .TRUE.
 SIGMA = 0.1
 NSW = 5

Al
 1.0000000000000000
     4.5000000000000000    0.0000000000000000    0.0000000000000000
     0.0000000000000000    4.5000000000000000    0.0000000000000000
     0.0000000000000000    0.0000000000000000    4.5000000000000000
   4
Cartesian
  0.0000000000000000  0.0000000000000000  0.0000000000000000
  0.0000000000000000  2.2500000000000000  2.2500000000000000
  2.2500000000000000  0.0000000000000000  2.2500000000000000
  2.2500000000000000  2.2500000000000000  0.0000000000000000

#+end_example

    -10.07430725
    
    Vasp calculation in /home-research/jkitchin/dft-book/bulk/Al-lda-vasp
    
    INCAR created by Atomic Simulation Environment
     ISIF = 7
     LCHARG = .FALSE.
     IBRION = 1
     EDIFFG = -0.001
     ISMEAR = 1
     LWAVE = .TRUE.
     SIGMA = 0.1
     NSW = 5
    
    Al
     1.0000000000000000
         4.5000000000000000    0.0000000000000000    0.0000000000000000
         0.0000000000000000    4.5000000000000000    0.0000000000000000
         0.0000000000000000    0.0000000000000000    4.5000000000000000
       4
    Cartesian
      0.0000000000000000  0.0000000000000000  0.0000000000000000
      0.0000000000000000  2.2500000000000000  2.2500000000000000
      2.2500000000000000  0.0000000000000000  2.2500000000000000
      2.2500000000000000  2.2500000000000000  0.0000000000000000



In [ ]:
from vasp import Vasp
calc = Vasp(label='bulk/Al-lda-vasp')
calc.view()
print([atoms.get_volume() for atoms in calc.traj])
print([atoms.get_potential_energy() for atoms in calc.traj])

    [91.124999999999986, 78.034123525818302, 72.328582812881763, 73.422437849114189, 73.368474506164134]
    [-9.58448747, -10.02992063, -10.07180132, -10.07429962, -10.07430725]

Now, the ASE way.
TODO



In [1]:
from vasp import Vasp
from ase.lattice import bulk
from ase.optimize import BFGS as QuasiNewton

Al = bulk('Al', 'fcc', a=4.5, cubic=True)

calc = Vasp(label='bulk/Al-lda-ase',
            xc='LDA',
            atoms=Al)

from ase.constraints import StrainFilter
sf = StrainFilter(Al)
qn = QuasiNewton(sf, logfile='relaxation.log')
qn.run(fmax=0.1, steps=5)
print('Stress:\n', calc.stress)
print('Al post ASE volume relaxation\n', calc.load_atoms().get_cell())
print(calc)

Now for a detailed comparison:



In [1]:
from vasp import Vasp

atoms = Vasp(label='bulk/Al-lda-vasp').load_atoms()

atoms2 = Vasp(label='bulk/Al-lda-ase').load_atoms()

import numpy as np

cellA = atoms.get_cell()
cellB = atoms2.get_cell()

print((np.abs(cellA - cellB) < 0.01).all())

False

    False

This could be handy if you want to use any of the optimizers in mod:ase.optimize in conjunction with mod:ase.constraints, which are more advanced than what is in VASP.

